# Phase 5: LLM Team Explainer

Generates natural-language explanations for the optimizer's fantasy XI using the Gemini API.

**Run cells top to bottom.** Cell 1 must run first (sets the OpenMP flag before torch/lightgbm import, which prevents the kernel crash on macOS).

In [1]:
# CELL 1 — MUST RUN FIRST, before any torch/lightgbm import
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import pickle
from lightgbm import LGBMRegressor

import torch
import torch.nn as nn

import pandas as pd
import numpy as np
import json
import time

print("imports ok")

imports ok


In [6]:
# CELL 2 — load data and config
df = pd.read_csv("../data/processed/player_match_features.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

credits = pd.read_csv("../data/processed/player_credits.csv")

with open("../models/ensemble_config.json") as f:
    ensemble_config = json.load(f)

sequence_features = ensemble_config["sequence_features"]
context_features = ensemble_config["context_features"]
SEQ_LEN = ensemble_config["seq_len"]

print("data ok:", df.shape, credits.shape)
print("seq/context/len:", len(sequence_features), len(context_features), SEQ_LEN)

data ok: (27909, 64) (811, 23)
seq/context/len: 11 21 7


/var/folders/3b/khlc6jhj47qcw7sj4kwtxp5m0000gn/T/ipykernel_27687/855124088.py:2: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/player_match_features.csv")


In [8]:
# CELL 3 — load LightGBM
with open("../models/lgbm_final.pkl", "rb") as f:
    lgbm = pickle.load(f)

lgbm_features = lgbm.feature_name_
print("lgbm ok,", len(lgbm_features), "features")

lgbm ok, 25 features


In [10]:
# CELL 4 — load LSTM
class CricketLSTM(nn.Module):
    def __init__(self, seq_features, context_features, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=seq_features, hidden_size=hidden_size,
                             num_layers=2, batch_first=True, dropout=0.3)
        self.fc1 = nn.Linear(hidden_size + context_features, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, seq, context):
        lstm_out, (hidden, cell) = self.lstm(seq)
        last_output = lstm_out[:, -1, :]
        combined = torch.cat([last_output, context], dim=1)
        x = self.fc1(combined)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x.squeeze(1)

device = torch.device("mps")
model = CricketLSTM(seq_features=len(sequence_features),
                     context_features=len(context_features),
                     hidden_size=32)
model.load_state_dict(torch.load("../models/lstm_final.pt", map_location=device))
model = model.to(device)
model.eval()

print("lstm ok")

lstm ok


In [11]:
# CELL 5 — optimizer functions
import pulp

def select_fantasy_team(match_pool, budget=100, min_wk=1, max_wk=4,
                          min_batters=3, max_batters=6,
                          min_bowlers=3, max_bowlers=6,
                          min_allrounders=1, max_allrounders=4,
                          max_per_team=7):

    players = match_pool["player"].tolist()
    points = dict(zip(match_pool["player"], match_pool["ensemble_pred"]))
    credits_map = dict(zip(match_pool["player"], match_pool["credit_value"]))
    roles = dict(zip(match_pool["player"], match_pool["role"]))
    teams = dict(zip(match_pool["player"], match_pool["team"]))

    prob = pulp.LpProblem("Fantasy_Team_Selection", pulp.LpMaximize)
    player_vars = {p: pulp.LpVariable(f"select_{p}", cat="Binary") for p in players}

    prob += pulp.lpSum([points[p] * player_vars[p] for p in players])
    prob += pulp.lpSum([player_vars[p] for p in players]) == 11
    prob += pulp.lpSum([credits_map[p] * player_vars[p] for p in players]) <= budget

    wk_players = [p for p in players if roles[p] == "wicketkeeper"]
    batter_players = [p for p in players if roles[p] == "batter"]
    bowler_players = [p for p in players if roles[p] == "bowler"]
    allrounder_players = [p for p in players if roles[p] == "allrounder"]

    prob += pulp.lpSum([player_vars[p] for p in wk_players]) >= min_wk
    prob += pulp.lpSum([player_vars[p] for p in wk_players]) <= max_wk
    prob += pulp.lpSum([player_vars[p] for p in batter_players]) >= min_batters
    prob += pulp.lpSum([player_vars[p] for p in batter_players]) <= max_batters
    prob += pulp.lpSum([player_vars[p] for p in bowler_players]) >= min_bowlers
    prob += pulp.lpSum([player_vars[p] for p in bowler_players]) <= max_bowlers
    prob += pulp.lpSum([player_vars[p] for p in allrounder_players]) >= min_allrounders
    prob += pulp.lpSum([player_vars[p] for p in allrounder_players]) <= max_allrounders

    for team_name in match_pool["team"].unique():
        team_players = [p for p in players if teams[p] == team_name]
        prob += pulp.lpSum([player_vars[p] for p in team_players]) <= max_per_team

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    status = pulp.LpStatus[prob.status]
    if status != "Optimal":
        print(f"Warning: solver status is {status}")
        return None

    selected = [p for p in players if player_vars[p].value() == 1]
    result = (match_pool[match_pool["player"].isin(selected)]
              .sort_values("ensemble_pred", ascending=False)
              .reset_index(drop=True))

    result["captain"] = False
    result["vice_captain"] = False
    result.loc[0, "captain"] = True
    result.loc[1, "vice_captain"] = True

    result["final_points"] = result["ensemble_pred"]
    result.loc[result["captain"], "final_points"] *= 2.0
    result.loc[result["vice_captain"], "final_points"] *= 1.5

    return result


def predict_and_select_team(match_df):
    match_df = match_df.copy()

    match_df["lgbm_pred"] = lgbm.predict(match_df[lgbm_features])

    lstm_preds_list = []
    for _, row in match_df.iterrows():
        history = df[(df["player"] == row["player"]) & (df["date"] < row["date"])].sort_values("date")
        hist_data = history[sequence_features].values
        if len(hist_data) < SEQ_LEN:
            pad_len = SEQ_LEN - len(hist_data)
            padding = np.zeros((pad_len, len(sequence_features)))
            seq = np.vstack([padding, hist_data])
        else:
            seq = hist_data[-SEQ_LEN:]

        context = row[context_features].values.astype(float)
        seq_t = torch.FloatTensor(seq).unsqueeze(0).to(device)
        context_t = torch.FloatTensor(context).unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(seq_t, context_t).cpu().numpy()[0]
        lstm_preds_list.append(pred)

    match_df["lstm_pred"] = lstm_preds_list
    match_df["ensemble_pred"] = 0.5 * match_df["lgbm_pred"] + 0.5 * match_df["lstm_pred"]

    match_pool = match_df.drop(columns=["role", "role_encoded"], errors="ignore").merge(
        credits[["player", "role", "credit_value"]], on="player", how="left"
    )

    team = select_fantasy_team(match_pool)
    return team, match_pool

print("optimizer functions defined")

optimizer functions defined


In [12]:
# CELL 6 — generate a team for a test match
test_2025 = df[df["date"].dt.year >= 2025]
sample_match_id = test_2025["match_id"].iloc[0]
sample_match = df[df["match_id"] == sample_match_id].copy()

team, match_pool = predict_and_select_team(sample_match)

print(team[["player", "team", "role", "credit_value", "ensemble_pred",
             "captain", "vice_captain", "final_points"]])
print("\nCredits used:", team["credit_value"].sum())
print("Total predicted points:", team["final_points"].sum())

            player                         team          role  credit_value  \
0        SP Narine        Kolkata Knight Riders    allrounder           9.0   
1          PD Salt  Royal Challengers Bengaluru        batter           9.2   
2          V Kohli  Royal Challengers Bengaluru        batter           9.4   
3          VR Iyer        Kolkata Knight Riders        batter           8.8   
4       RM Patidar  Royal Challengers Bengaluru        batter           9.4   
5        Q de Kock        Kolkata Knight Riders  wicketkeeper           9.3   
6        AM Rahane        Kolkata Knight Riders        batter           8.5   
7   LS Livingstone  Royal Challengers Bengaluru        batter           8.3   
8     Harshit Rana        Kolkata Knight Riders        bowler           8.6   
9         CV Varun        Kolkata Knight Riders        bowler           8.7   
10      SH Johnson        Kolkata Knight Riders        bowler           8.1   

    ensemble_pred  captain  vice_captain  final_poi

In [1]:
# CELL 7 — Gemini client with retry/backoff
from dotenv import load_dotenv
load_dotenv()

from google import genai
client = genai.Client()

import os
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def call_llm(prompt, model_name="llama-3.3-70b-versatile", max_retries=4):
    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=600
            )
            return response.choices[0].message.content
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                raise RuntimeError(f"Groq rate limit hit: {e}") from e
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"Attempt {attempt+1} failed, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

rag_explanation = call_llm(rag_prompt)
print(rag_explanation)


ModuleNotFoundError: No module named 'groq'

In [15]:
# CELL 8 — build the explanation prompt
def build_explanation_prompt(team_df, team1, team2, venue):
    lines = []
    for _, row in team_df.iterrows():
        tag = ""
        if row["captain"]:
            tag = " (CAPTAIN, 2x points)"
        elif row["vice_captain"]:
            tag = " (VICE-CAPTAIN, 1.5x points)"
        lines.append(
            f"- {row['player']} ({row['team']}, {row['role']}) "
            f"- predicted {row['ensemble_pred']:.1f} pts, {row['credit_value']} credits{tag}"
        )

    team_list_str = "\n".join(lines)
    total_credits = team_df["credit_value"].sum()
    total_points = team_df["final_points"].sum()

    prompt = f"""You are a cricket fantasy analyst. Explain this fantasy XI selection for {team1} vs {team2} at {venue}.

Selected team ({total_credits:.1f}/100 credits used, {total_points:.1f} total projected points including captain multipliers):
{team_list_str}

Write a clear 150-200 word explanation covering:
1. Why the captain and vice-captain were chosen
2. The team's batting/bowling balance
3. One or two standout picks worth highlighting
4. A brief, honest note that these are model-based projections, not guarantees

Write in plain paragraphs. Do not use markdown, bullet points, or headers."""

    return prompt


prompt = build_explanation_prompt(
    team,
    sample_match["team"].iloc[0],
    sample_match["opposition"].iloc[0],
    sample_match["venue"].iloc[0]
)
print(prompt)

You are a cricket fantasy analyst. Explain this fantasy XI selection for Royal Challengers Bengaluru vs Kolkata Knight Riders at Eden Gardens, Kolkata.

Selected team (97.3/100 credits used, 420.5 total projected points including captain multipliers):
- SP Narine (Kolkata Knight Riders, allrounder) - predicted 46.3 pts, 9.0 credits (CAPTAIN, 2x points)
- PD Salt (Royal Challengers Bengaluru, batter) - predicted 39.2 pts, 9.2 credits (VICE-CAPTAIN, 1.5x points)
- V Kohli (Royal Challengers Bengaluru, batter) - predicted 39.2 pts, 9.4 credits
- VR Iyer (Kolkata Knight Riders, batter) - predicted 37.8 pts, 8.8 credits
- RM Patidar (Royal Challengers Bengaluru, batter) - predicted 37.1 pts, 9.4 credits
- Q de Kock (Kolkata Knight Riders, wicketkeeper) - predicted 35.2 pts, 9.3 credits
- AM Rahane (Kolkata Knight Riders, batter) - predicted 34.6 pts, 8.5 credits
- LS Livingstone (Royal Challengers Bengaluru, batter) - predicted 34.1 pts, 8.3 credits
- Harshit Rana (Kolkata Knight Riders, bo

In [19]:
# CELL 9 — generate the explanation
explanation = call_gemini(prompt)
print(explanation)

Attempt 1 failed, retrying in 1s...
Attempt 2 failed, retrying in 2s...
Sunil Narine earns the captain's armband because his dual-threat capability at Eden Gardens offers unmatched fantasy utility, combining explosive opening batting with reliable four-over mystery spin on a traditionally responsive pitch. Phil Salt takes the vice-captain role to capitalize on his blistering powerplay scoring and wicketkeeping dismissals, giving this fantasy lineup two rapid point-accumulators right at the top of the order.

Structurally, the squad leans heavily toward batting, featuring seven proven run-makers to maximize points in anticipated high-scoring Eden conditions. The bowling responsibilities are concentrated within Kolkata's frontline attack through Harshit Rana, Varun Chakravarthy, and Spencer Johnson, aiming to sweep wickets when Bengaluru takes risks against them.

Liam Livingstone represents a standout differential pick, offering raw boundary-hitting power in the middle overs and useful 

In [20]:
print("Models supporting embedContent:\n")
for m in client.models.list():
    if "embedContent" in m.supported_actions:
        print(m.name)

Models supporting embedContent:

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [22]:
from typing import List, Dict

def fetch_from_manual(text: str, source: str = "manual", date: str = None) -> List[Dict]:
    """Wraps a pasted article into our standard document format."""
    return [{
        "text": text,
        "source": source,
        "date": date
    }]


# paste a real pre-match preview here
sample_preview = """
Not since Brendon McCullum's 158* has the IPL opening night promised a storm quite like the one that's in store for the 2025 season.
Quite literally too. There's an Orange Alert in place for the match-day in Kolkata, which could mean more rain than there's been in the lead-up, stronger winds, and even a thunderstorm. But if there's one thing the city knows how to do, it's turning chaos into celebration. Rain never stopped a Pujo and it won't stop thousands from streaming into Eden Gardens, where the IPL opening ceremony returns for the first time since 2015.
But that said, this is cricket and is far more of a slave to the whims of the sky. Sometimes, all it takes is one stubborn wet patch to derail the night but let's not speak of such things. The big covers at Eden exist for a reason, and the groundsmen are already busy figuring out how to shield the outfield from rain. And their covers from the heels of dancers.
The opening ceremony is set to begin 45 minutes before the toss but expect real frills and thrills once the cricket starts. There's always an extra edge when KKR and RCB face off, no matter where in the world. And this is Eden, a ground with a penchant for both magic and meltdowns. It's where RCB were bowled out for 49, still the lowest total in IPL history. It's where Sunil Narine is most frequently seen batting like a man possessed, and where a seething Virat Kohli once lost his cool over a no-ball height call. Add to that Andre Russell's fireworks, Varun Chakaravarthy's mystery, and the baggage of past drama, you begin to get a sense of how much this fixture carries.
What makes it even more intriguing is that Eden is barely a fortress anymore. Since 2023, KKR have won just 50% of their home games, the exact same as RCB's record at the Chinnaswamy. That is part of what makes this match-up compelling: two dangerous but vulnerable sides, brimming with firepower, both eager to launch a new three-year cycle on the right note. KKR boast the stronger spin unit; RCB, the more experienced pace attack. The question isn't who has the better side; it's whose strengths will rise to the surface when the music fades and the cricket begins.
No wonder Andy Flower, RCB's head coach, called the prospect of playing KKR at Eden "daunting... for Kolkata," before breaking into a wide grin.
And that's the thing about opening nights. They rarely follow the script. The sky may yet have its say but if it doesn't for long enough, this could be one of those nights Eden doesn't forget in a hurry.
When: KKR v RCB, Match 1, on March 22, 2025 at 7:30 pm IST
Where: Eden Gardens, Kolkata
What to expect: A wet ball, which could be one of the central characters of the night, be it from the dew, the forecasted rain or a bit of saliva slipping its way back into cricket. Also expect ample airtime for Eden Gardens' famously large white covers.
The night temperature is predicted to dip by 4-5 degrees, which could mean heavier dew and the new playing conditions around ball-change in the second innings could be put to test immediately.
And then there's the pitch, which is expected to be flat, fast and full of runs. Since the IPL returned to the home-away format in 2023, Eden Gardens has been the highest-scoring venue in the league, with a staggering scoring rate of 9.98 runs per over, the most for any ground with at least three games.
Recent head-to-head: Since the start of the UAE leg of IPL 2021, KKR have dominated this rivalry, winning six of their last seven matches against RCB. But things even out at Eden Gardens, where the last four meetings between these two have produced two wins each.
Team Watch
Kolkata Knight Riders
Injuries/availability: Anrich Nortje returns after a back injury ruled him out of the SA20 and the Champions Trophy but, as rain washed out both teams' training sessions on match eve, there was little to note about his match readiness. The bigger question, though, is whether there's a spot for him in the XI at all. You'd think Spencer Johnson, with his left-arm angle and the variety he offers, might edge ahead especially considering Nortje's modest IPL record outside of the UAE.
Tactics & matchups: Mitchell Starc's presence last season allowed KKR to backload the overs from Chakaravarthy, who bowled just four in the PowerPlay in 2024 as compared to 16 in 2023. With no Starc this time and limited experience in the new-ball department, KKR may be forced to frontload their spinners once again to compensate for the lack of early-impact seam options.
Probable XII: Sunil Narine, Quinton de Kock (wk), Ajinkya Rahane (c), Venkatesh Iyer, Angkrish Raghuvanshi, Rinku Singh, Andre Russell, Ramandeep Singh, Spencer Johnson, Vaibhav Arora, Harshit Rana, Varun Chakaravarthy
Royal Challengers Bengaluru
Injuries/availability: Jacob Bethell and Josh Hazlewood underwent fitness tests two days out from the match and both appeared to hold their own during practice. Hazlewood bowled a short spell, not quite at full tilt but with a decent rhythm and a full run-up. Head coach Andy Flower also sounded confident about the shape of his bowling attack, one that notably included Hazlewood.
Tactics & matchups: Spin could be the key against an RCB line-up that can be tied down by quality spin. There are two notable exceptions in Rajat Patidar, whose strike rate against spin shoots up nearly 40 points to 197, and Jitesh Sharma, who strikes at around 145 against both pace and spin with equal comfort. Kohli, on the other hand, has been far more circumspect despite having a breakout season against spin last year. He strikes at just 103 against Narine and has fallen to him four times. Against Chakaravarthy, the number drops further to 102.5, though he's only been dismissed once.
Probable XII: Phil Salt, Virat Kohli, Devdutt Padikkal, Rajat Patidar (c), Liam Livingstone, Jitesh Sharma (wk), Tim David, Krunal Pandya, Bhuvneshwar Kumar, Josh Hazlewood, Yash Dayal, Suyash Sharma/Rasikh Dar Salam
Did you know?
This is the second instance of KKR playing RCB in an IPL season opener after having done so in the inaugural edition in 2008
There have been 12 200+ totals at Eden in just the last two seasons, compared to only 10 such scores until 2022
Ajinkya Rahane was both the leading run-getter and Player of the Tournament in SMAT 2024-25 in Mumbai's title-winning run
What they said
"I did well in the last tournament but still IPL is a different beast and I very well know what's coming my way and I have to be on my toes." - Varun Chakaravarthy
"I hope it is an El Clasico tomorrow night. That'll be a brilliant way to start IPL 2025!" - Andy Flower
Squads:
Royal Challengers Bengaluru Squad: Virat Kohli, Philip Salt, Devdutt Padikkal, Rajat Patidar(c), Jitesh Sharma(w), Liam Livingstone, Tim David, Krunal Pandya, Bhuvneshwar Kumar, Josh Hazlewood, Yash Dayal, Swapnil Singh, Lungi Ngidi, Romario Shepherd, Manoj Bhandage, Rasikh Dar Salam, Nuwan Thushara, Jacob Bethell, Suyash Sharma, Mohit Rathee, Swastik Chikara, Abhinandan Singh
Kolkata Knight Riders Squad: Quinton de Kock(w), Sunil Narine, Ajinkya Rahane(c), Angkrish Raghuvanshi, Venkatesh Iyer, Rinku Singh, Andre Russell, Ramandeep Singh, Harshit Rana, Varun Chakaravarthy, Spencer Johnson, Vaibhav Arora, Rahmanullah Gurbaz, Manish Pandey, Moeen Ali, Anrich Nortje, Rovman Powell, Anukul Roy, Mayank Markande, Chetan Sakariya, Luvnith Sisodia
"""

docs = fetch_from_manual(sample_preview, source="manual_paste", date="2025-03-22")
print(f"{len(docs)} document(s), {len(docs[0]['text'])} characters")

1 document(s), 7377 characters


In [24]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50, min_chunk_words: int = 50) -> List[str]:
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        
        if len(chunk_words) < min_chunk_words and chunks:
            # append leftovers to the last chunk instead of orphaning them
            chunks[-1] = chunks[-1] + " " + " ".join(chunk_words)
            break
            
        chunks.append(" ".join(chunk_words))
        start += chunk_size - overlap
    
    return chunks


def chunk_documents(docs: List[Dict], chunk_size: int = 300, overlap: int = 50) -> List[Dict]:
    """Chunk every document, preserving its metadata on each chunk."""
    all_chunks = []
    for doc in docs:
        for i, chunk in enumerate(chunk_text(doc["text"], chunk_size, overlap)):
            all_chunks.append({
                "text": chunk,
                "source": doc["source"],
                "date": doc["date"],
                "chunk_index": i
            })
    return all_chunks


chunks = chunk_documents(docs)
print(f"{len(chunks)} chunks created\n")
for c in chunks:
    print(f"--- chunk {c['chunk_index']} ({len(c['text'].split())} words) ---")
    print(c["text"][:200] + "...\n")

5 chunks created

--- chunk 0 (300 words) ---
Not since Brendon McCullum's 158* has the IPL opening night promised a storm quite like the one that's in store for the 2025 season. Quite literally too. There's an Orange Alert in place for the match...

--- chunk 1 (300 words) ---
and where a seething Virat Kohli once lost his cool over a no-ball height call. Add to that Andre Russell's fireworks, Varun Chakaravarthy's mystery, and the baggage of past drama, you begin to get a ...

--- chunk 2 (300 words) ---
of saliva slipping its way back into cricket. Also expect ample airtime for Eden Gardens' famously large white covers. The night temperature is predicted to dip by 4-5 degrees, which could mean heavie...

--- chunk 3 (300 words) ---
backload the overs from Chakaravarthy, who bowled just four in the PowerPlay in 2024 as compared to 16 in 2023. With no Starc this time and limited experience in the new-ball department, KKR may be fo...

--- chunk 4 (252 words) ---
Phil Salt, Virat Kohli

In [37]:
import os
def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT"):
    """Embed a list of texts one at a time. Returns array of shape (n_texts, embedding_dim)."""
    vectors = []
    
    for idx, text in enumerate(texts):
        last_error = None
        for attempt in range(4):
            try:
                response = client.models.embed_content(
                    model="models/gemini-embedding-2",
                    contents=text,
                    config=types.EmbedContentConfig(task_type=task_type)
                )
                vectors.append(response.embeddings[0].values)
                break
            except Exception as e:
                last_error = e
                if attempt < 3:
                    wait = 2 ** attempt
                    print(f"Text {idx}, attempt {attempt+1} failed: {e}")
                    time.sleep(wait)
        else:
            raise RuntimeError(f"Failed to embed text {idx}. Last error: {last_error}")
    
    return np.array(vectors)




chunk_texts = [c["text"] for c in chunks]

EMBED_CACHE_PATH = "../data/processed/chunk_vectors.npy"

if os.path.exists(EMBED_CACHE_PATH):
    chunk_vectors = np.load(EMBED_CACHE_PATH)
    print("Loaded cached embeddings:", chunk_vectors.shape)
else:
    chunk_vectors = embed_texts(chunk_texts, task_type="RETRIEVAL_DOCUMENT")
    np.save(EMBED_CACHE_PATH, chunk_vectors)
    print("Computed and cached embeddings:", chunk_vectors.shape)

Computed and cached embeddings: (5, 3072)


In [38]:
def cosine_similarity(query_vec: np.ndarray, doc_vecs: np.ndarray) -> np.ndarray:
    """Cosine similarity between one query vector and many document vectors."""
    query_norm = query_vec / np.linalg.norm(query_vec)
    doc_norms = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)
    return doc_norms @ query_norm


def retrieve(query: str, chunks: List[Dict], chunk_vectors: np.ndarray, top_k: int = 3) -> List[Dict]:
    """Embed the query, find the most similar chunks, return them with scores."""
    query_vec = embed_texts([query], task_type="RETRIEVAL_QUERY")[0]
    scores = cosine_similarity(query_vec, chunk_vectors)
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        result = chunks[idx].copy()
        result["score"] = float(scores[idx])
        results.append(result)
    
    return results


# test it
results = retrieve("What are the pitch conditions expected to be like?", chunks, chunk_vectors, top_k=2)

for r in results:
    print(f"--- score {r['score']:.3f} (chunk {r['chunk_index']}) ---")
    print(r["text"][:300] + "...\n")

--- score 0.680 (chunk 2) ---
of saliva slipping its way back into cricket. Also expect ample airtime for Eden Gardens' famously large white covers. The night temperature is predicted to dip by 4-5 degrees, which could mean heavier dew and the new playing conditions around ball-change in the second innings could be put to test i...

--- score 0.665 (chunk 0) ---
Not since Brendon McCullum's 158* has the IPL opening night promised a storm quite like the one that's in store for the 2025 season. Quite literally too. There's an Orange Alert in place for the match-day in Kolkata, which could mean more rain than there's been in the lead-up, stronger winds, and ev...



In [39]:
def build_rag_explanation_prompt(team_df, team1, team2, venue, retrieved_chunks):
    lines = []
    for _, row in team_df.iterrows():
        tag = ""
        if row["captain"]:
            tag = " (CAPTAIN, 2x points)"
        elif row["vice_captain"]:
            tag = " (VICE-CAPTAIN, 1.5x points)"
        lines.append(
            f"- {row['player']} ({row['team']}, {row['role']}) "
            f"- predicted {row['ensemble_pred']:.1f} pts, {row['credit_value']} credits{tag}"
        )
    team_list_str = "\n".join(lines)

    context_str = "\n\n".join([
        f"[Source: {c['source']}, relevance {c['score']:.2f}]\n{c['text']}"
        for c in retrieved_chunks
    ])

    total_credits = team_df["credit_value"].sum()
    total_points = team_df["final_points"].sum()

    prompt = f"""You are a cricket fantasy analyst explaining a model-generated fantasy XI for {team1} vs {team2} at {venue}.

MATCH CONTEXT (from recent pre-match reporting):
{context_str}

MODEL-SELECTED TEAM ({total_credits:.1f}/100 credits, {total_points:.1f} projected points):
{team_list_str}

Write a 200-250 word explanation that:
1. Explains the captain and vice-captain choices
2. Connects the team selection to the actual match conditions described in the context above, especially pitch, weather, and dew
3. Flags any tension between what the model predicts and what the match context suggests
4. Notes honestly that the model works from historical data and cannot account for late team news

Only reference conditions actually mentioned in the context. Do not invent details. Write in plain paragraphs, no markdown."""

    return prompt


# retrieve context relevant to team selection
context_query = "pitch conditions, weather, dew, player form, team news"
retrieved = retrieve(context_query, chunks, chunk_vectors, top_k=3)

rag_prompt = build_rag_explanation_prompt(
    team,
    sample_match["team"].iloc[0],
    sample_match["opposition"].iloc[0],
    sample_match["venue"].iloc[0],
    retrieved
)

rag_explanation = call_gemini(rag_prompt)
print(rag_explanation)

Attempt 1 failed, retrying in 1s...
Attempt 2 failed, retrying in 2s...
Attempt 3 failed, retrying in 4s...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 36.241169903s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.8-flash'},

In [40]:
print(rag_prompt)

You are a cricket fantasy analyst explaining a model-generated fantasy XI for Royal Challengers Bengaluru vs Kolkata Knight Riders at Eden Gardens, Kolkata.

MATCH CONTEXT (from recent pre-match reporting):
[Source: manual_paste, relevance 0.68]
of saliva slipping its way back into cricket. Also expect ample airtime for Eden Gardens' famously large white covers. The night temperature is predicted to dip by 4-5 degrees, which could mean heavier dew and the new playing conditions around ball-change in the second innings could be put to test immediately. And then there's the pitch, which is expected to be flat, fast and full of runs. Since the IPL returned to the home-away format in 2023, Eden Gardens has been the highest-scoring venue in the league, with a staggering scoring rate of 9.98 runs per over, the most for any ground with at least three games. Recent head-to-head: Since the start of the UAE leg of IPL 2021, KKR have dominated this rivalry, winning six of their last seven matches